# Session 2 - control detectors + ablations


**CPU, 2-4 h.** Runs the four synthetic control detectors, the operator ablations,
and ground-truth localization.

This session validates the instrument. The controls are constructed with known
faithfulness, so their ordering on **real frames** is a check on the masks and the
data: the audit proceeds to the GPU sessions only if that ordering holds.

Inputs: `<your-code-dataset>` and `cca-s1-parsed`. Accelerator **None**.

In [ ]:
SESSION = "S2 controls"

# ============================== CONFIG ==============================
CONTROLS    = "adaptive_oracle,fixed_oracle:mouth,confabulator,dummy"
N_PROC      = 4          # one shard per CPU core
BLEND       = "poisson"
DILATE      = 3
JPEG_Q      = 90
INPAINT     = "telea"    # inpainting pipeline check ("" = off)
SPLICE_FLOOR = True      # operator validity check on authentic frames; keep on

# operator ablations. 0 samples = skip.
ABL_SAMPLES = 600
ABL_BLENDS  = ["feather", "hard"]
ABL_DILATE  = [0, 7, 11]
ABL_NO_REENCODE = True

MAIN_BUDGET_MIN = 300    # controls stop cleanly at 5 h
ABL_BUDGET_MIN  = 150    # ablations get a further 2.5 h -> 7.5 h total
MATERIALISE_SAMPLES = 30 # PNG condition images for figures
SEED = 0

In [ ]:
# ---------------------------------------------------------------- BOOT
# Locates the code dataset wherever it is mounted, puts it on sys.path,
# prints the attached inputs, and records provenance.  The search is by
# file name, so the dataset's mount name does not matter.
import os, sys, subprocess, json, time

def _find_code():
    for root in ("/kaggle/input", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, files in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if os.path.basename(dirpath) == "ccaudit" and "kaggle_utils.py" in files:
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not find the ccaudit package.\n"
        "Add Input -> your code dataset (<your-code-dataset>), and check that "
        "the preview shows ccaudit/kaggle_utils.py at the top level.")

CODE = _find_code()
if CODE not in sys.path:
    sys.path.insert(0, CODE)
# Child processes do not inherit sys.path.  Every `python -m ccaudit.<module>`
# below runs as a subprocess, so the code directory must be on PYTHONPATH.
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
from ccaudit import kaggle_utils as KU
from ccaudit import common as C

OUT = KU.work_dir("audit")
TMP = KU.temp_dir()
os.environ["HF_HOME"] = KU.temp_dir("hf")          # model weights stay out of /kaggle/working
os.environ["TOKENIZERS_PARALLELISM"] = "false"
KU.session_header(SESSION, OUT)
print("code:", CODE)

In [ ]:
# ------------------------------------------------------- SELF TEST (always)
# The self-test suite runs in under a minute and needs no dataset.  Each check
# corresponds to a failure mode that would produce plausible-looking but
# incorrect numbers, so a failure here invalidates everything that follows.
rc = KU.sh(f"{sys.executable} {CODE}/scripts/selftest.py", check=False)
if rc != 0:
    raise SystemExit("SELF TEST FAILED -- inspect the failures above before proceeding.")

In [ ]:
# ------------------------------------------------------- locate session 1
INDEX = KU.find_parsed_index()
if not INDEX:
    raise SystemExit(
        "Session 1 output not found. Add Input -> your `cca-s1-parsed` dataset "
        "(or Your Work -> Notebooks -> the session 1 notebook).")
print("INDEX =", INDEX)
recs, meta = C.load_index(INDEX)
print(f"{len(recs)} samples, vocab={meta.get('vocab')}, "
      f"crop={meta.get('crop_size')}")
VOCAB = meta.get("vocab", "face8")

In [ ]:
# ==================== CONTROL DETECTORS (sharded over cores) ====================
floor = " --splice-floor" if SPLICE_FLOOR else ""
inp   = f" --inpaint {INPAINT}" if INPAINT else ""
cmds, logs = [], []
for i in range(N_PROC):
    logs.append(f"{OUT}/logs/controls_{i}.log")
    cmds.append(
        f'{sys.executable} -m ccaudit.m5_runner --index "{INDEX}" '
        f'--detector "{CONTROLS}" --out "{OUT}/run_controls" --tag main '
        f'--blend {BLEND} --dilate {DILATE} --jpeg-q {JPEG_Q} '
        f'--shard {i}/{N_PROC} --device cpu --vocab {VOCAB} '
        f'--time-budget-min {MAIN_BUDGET_MIN}{floor}{inp}')
KU.run_parallel(cmds, logs=logs, check=False)
KU.tail(logs[0], 12)

In [ ]:
# ==================== OPERATOR ABLATIONS ====================
# Each cell of the grid is a separate tag, so Session 11 reports them as rows.
if ABL_SAMPLES:
    jobs = []
    for b in ABL_BLENDS:
        jobs.append((f"blend_{b}", f"--blend {b} --dilate {DILATE} --jpeg-q {JPEG_Q}"))
    for d in ABL_DILATE:
        jobs.append((f"dilate_{d}", f"--blend {BLEND} --dilate {d} --jpeg-q {JPEG_Q}"))
    if ABL_NO_REENCODE:
        jobs.append(("no_reencode", f"--blend {BLEND} --dilate {DILATE} --jpeg-q 0"))
    per = max(1, ABL_BUDGET_MIN // max(1, len(jobs)))
    for tag, flags in jobs:
        cmds, logs = [], []
        for i in range(N_PROC):
            logs.append(f"{OUT}/logs/abl_{tag}_{i}.log")
            cmds.append(
                f'{sys.executable} -m ccaudit.m5_runner --index "{INDEX}" '
                f'--detector "{CONTROLS}" --out "{OUT}/run_ablation" '
                f'--tag {tag} {flags} --limit-samples {ABL_SAMPLES} '
                f'--seed {SEED} --shard {i}/{N_PROC} --device cpu '
                f'--vocab {VOCAB} --time-budget-min {per}')
        print(C.banner(f"ablation: {tag}"))
        KU.run_parallel(cmds, logs=logs, check=False)
else:
    print("ablations skipped (ABL_SAMPLES = 0)")

In [ ]:
# ==================== GROUND-TRUTH LOCALIZATION (m10) ====================
KU.sh(f'{sys.executable} -m ccaudit.m10_localization --index "{INDEX}" '
      f'--out "{OUT}/loc"', check=False, log=f"{OUT}/logs/m10.log")

In [ ]:
# ==================== METRICS + REPORT ====================
raw = f"{OUT}/run_controls,{OUT}/run_ablation"
for extra, out in ((f'--localization "{OUT}/loc/localization.json" --by method',
                    f"{OUT}/metrics"),
                   ("--coarse", f"{OUT}/metrics")):
    KU.sh(f'{sys.executable} -m ccaudit.m6_metrics --raw "{raw}" '
          f'--out "{out}" --vocab {VOCAB} {extra}', check=False,
          log=f"{OUT}/logs/m6.log")
KU.sh(f'{sys.executable} -m ccaudit.m7_report '
      f'--metrics "{OUT}/metrics/metrics.json" '
      f'--coarse "{OUT}/metrics/metrics_coarse.json" '
      f'--by-method "{OUT}/metrics/metrics_by_method.json" '
      f'--raw "{OUT}/run_controls" --out "{OUT}/report_controls.html" '
      f'--figures-dir "{OUT}/figures" '
      f'--title "Counterfactual citation audit - control detectors"', check=False)

In [ ]:
# ---------------- INSTRUMENT CHECK ----------------
# The controls are constructed so that adaptive_oracle is faithful, the
# confabulator detects well but cites incorrectly, and fixed_oracle does not
# exceed adaptive_oracle on reverse citation.  The audit proceeds to the GPU
# sessions only if that ordering holds on real frames.
res = C.load_json(f"{OUT}/metrics/metrics.json", {}).get("results", [])
by = {r["detector"]: r for r in res}
print(f"{'detector':24s}{'AUC':>8}{'FS':>10}{'CR-prior':>10}  verdict")
for d, r in sorted(by.items()):
    v = ("FAITHFUL" if r.get("faithful") else
         "fwd only" if r.get("faithful_forward") else "UNFAITHFUL")
    print(f"{d:24s}{r.get('AUC',float('nan')):>8.3f}"
          f"{r.get('FS',float('nan')):>10.4f}"
          f"{r.get('CR_minus_prior',float('nan')):>10.3f}  {v}")

problems = []
ad = by.get("adaptive_oracle")
cf = by.get("confabulator")
fx = next((v for k, v in by.items() if k.startswith("fixed_oracle")), None)
if ad and not ad.get("faithful_forward"):
    problems.append("adaptive_oracle is not faithful in the forward direction")
if cf and abs(cf.get("FS", 0)) > 0.5 * abs((ad or {}).get("FS", 1)):
    problems.append("confabulator's FS is not clearly below adaptive_oracle's")
if cf and cf.get("AUC", 0) < 0.7:
    problems.append("confabulator's AUC is low: by construction it should detect "
                    "well and explain badly")
if fx and fx.get("CR_minus_prior", 0) > (ad or {}).get("CR_minus_prior", 1):
    problems.append("fixed_oracle exceeds adaptive_oracle on reverse citation")
for r in res:
    fp = r.get("floor_p_mean")
    if fp is not None and fp == fp and fp > 0.5:
        problems.append(f"{r['detector']}: floor p = {fp:.2f}; the operator "
                        f"may be introducing forgery evidence on authentic frames")
print()
if problems:
    print("!! INSTRUMENT PROBLEMS -- resolve these before the GPU sessions:")
    for p in problems:
        print("   -", p)
else:
    print("Instrument validation passed. Proceed to the GPU sessions.")

In [ ]:
# ---------------- condition images for figures ----------------
if MATERIALISE_SAMPLES:
    from ccaudit import m3_splice as M3
    from ccaudit import regions as R
    R.set_vocab(VOCAB)
    p = M3.build(INDEX, f"{OUT}/cond_sample", n_samples=MATERIALISE_SAMPLES,
                 dilate=DILATE, mode=BLEND, jpeg_q=JPEG_Q, with_floor=True,
                 inpaint_method=INPAINT or "")
    print("condition images ->", p)

try:
    from IPython.display import HTML, display
    if os.path.exists(f"{OUT}/report_controls.html"):
        display(HTML(open(f"{OUT}/report_controls.html").read()))
except Exception as exc:
    print(f"(inline report unavailable: {exc}); download "
          f"{OUT}/report_controls.html from the Output panel")

In [ ]:
NEXT_STEP = """1. Save Version -> Save & Run All.
2. Output tab -> New Dataset -> `cca-s2-controls`.
3. If the instrument check above reported problems, resolve them first. The
   usual causes are a bad parse (inspect Session 1's overlay) or a mirror
   whose "fake" clips are not actually paired with the originals.
4. Otherwise continue with Session 3 (03_gpu_vlm_smoke.ipynb) to measure VLM
   throughput before committing GPU time."""

# ----------------------------------------------------------- WRAP UP
KU.disk_report()
print(C.banner("NEXT STEP"))
print(NEXT_STEP)